# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [4]:
import pandas as pd
import plotly.express as px


df = pd.read_csv('global_energy_mix.csv')


fossil_sources = ['Coal', 'Oil', 'Natural Gas']
df_fossil = df[df['Source'].isin(fossil_sources)].copy()


color_map = {
    'Coal': '#440154',
    'Oil': '#21918c',
    'Natural Gas': '#fde725'
}


top_country = df_fossil.groupby('Country')['TWh'].sum().idxmax()
top_region = df_fossil[df_fossil['Country'] == top_country]['Region'].iloc[0]


fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=color_map,
    title=f"<b>Insight: {top_country} in {top_region} is the most fossil-dependent entity by TWh</b>"
)


fig.update_traces(
    textinfo="label+value",
    hovertemplate='<b>%{label}</b><br>Consumption: %{value} TWh',
    marker=dict(line=dict(width=1, color='white'))
)


fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    treemapcolorway=["#D3D3D3"] # Default grey for parent levels
)


fig.show()

## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [5]:
# Task 2
# YOUR CODE HERE
import plotly.express as px

df = px.data.tips()


df_agg = df.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()


color_map = {
    'Yes': '#0072B2',  # Smoker: Blue
    'No': '#D55E00'    # Non-Smoker: Orange
}


top_group = df_agg.loc[df_agg['total_bill'].idxmax()]
insight_title = f"Insight: Most spending occurs on {top_group['day']} during {top_group['time']} by {'non-smokers' if top_group['smoker']=='No' else 'smokers'}"


fig = px.sunburst(
    df_agg,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=color_map,
    title=f"<b>{insight_title}</b>"
)


fig.update_traces(
    textinfo="label+percent parent",
    hovertemplate='<b>%{label}</b><br>Total Bill Sum: $%{value:.2f}'
)


fig.update_layout(
    margin=dict(t=80, l=0, r=0, b=0),
    sunburstcolorway=["#D3D3D3"]  # Sets the default color (for parents) to Grey
)

fig.show()

## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [6]:
# Task 3 — charts
# YOUR CODE HERE
import pandas as pd
import plotly.express as px


df = pd.read_csv('global_energy_mix.csv')


low_carbon_sources = ['Nuclear', 'Hydro']
df_lc = df[df['Source'].isin(low_carbon_sources)].copy()

df_agg = df_lc.groupby('Country')['TWh'].sum().reset_index()


df_agg['All'] = 'Low-carbon'


df_bar = df_agg.sort_values(by='TWh', ascending=True) # Ascending=True for horizontal bar sorting


top_country = df_agg.loc[df_agg['TWh'].idxmax()]

# =====================================
# TREEMAP: Comparative Volume
# =====================================
fig_treemap = px.treemap(
    df_agg,
    path=['All', 'Country'],
    values='TWh',
    title="<b>Low-carbon Energy Distribution (Nuclear + Hydro)</b>",
    color_discrete_sequence=['#D3D3D3'] # Grey for the parent, children inherit
)
fig_treemap.update_traces(textinfo="label+value")


# =====================================
# BAR CHART: Ranking and Leading Country
# =====================================
fig_bar = px.bar(
    df_bar,
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    title=f"<b>Insight: {top_country['Country']} leads in Low-carbon energy with {top_country['TWh']:,.0f} TWh</b>",
    color_discrete_sequence=['#56B4E9'] # CVD-safe Blue
)
fig_bar.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig_bar.update_layout(xaxis_title="Energy Consumption (TWh)", yaxis_title="Country")


fig_treemap.show()
fig_bar.show()